In [ ]:
from PosSelect_Functions_Old import *
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)

def quantile_normalize_phylop(v, vv):
    #Upsample the lower one, maximum of one duplicate per entry added
    v_samp = v.sample(frac = (vv.shape[0] - v.shape[0])/v.shape[0], replace = True)
    v_samp.index = range(v.shape[0], v.shape[0] + v_samp.shape[0])
    
    v_new = pd.concat([v, v_samp]).sort_values("PhyloP447")

    #Make new dataframes and join
    vv_new = vv.copy().sort_values("PhyloP447")
    #v_new.columns = ["v0", "v1", "v2", "v3", "v4"]
    #vv_new.columns = ["vv0", "vv1", "vv2", "vv3"]
    v_new.index = list(range(v_new.shape[0]))
    vv_new.index = list(range(vv_new.shape[0]))
    v_new.columns = [x + " poly" for x in v_new.columns]
    vv_new.columns = [x + " fixed" for x in vv_new.columns]
    v_quant = v_new.join(vv_new)

    #Make dataframe to normalize
    df = v_quant[["PhyloP447 poly", "PhyloP447 fixed"]].copy()
    rank_mean = df.stack().groupby(df.rank(method='first').stack().astype(int)).mean()
    print(rank_mean)
    df = df.rank(method='min').stack().astype(int).map(rank_mean).unstack()
    df.columns = ["PhyloP447 poly", "PhyloP447 fixed"]
    new_v1 = np.mean(df, axis = 1)
    df["Statistic"] = new_v1
    df = df[["Statistic"]].copy()
    df = df.join(v_quant)
    return df

hpo = pd.read_csv("../DPSC_CNCC/HPO_AccelEvol_Input.txt", sep= "\t")
d_HPO = {}

for index, row in hpo.iterrows():
    d_HPO[row["Term"]] = row["Genes"].split(";")

gobp = pd.read_csv("../DPSC_CNCC/GOBP_AccelEvol_Input.txt", sep= "\t")
d_BP = {}

for index, row in gobp.iterrows():
    if "ucus" in row["Term"]:
        print(row["Term"])
    d_BP[row["Term"]] = row["Genes"].split(";")


In [ ]:

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from time import perf_counter

# -----------------------------
# Settings
# -----------------------------
N_PERMUTATIONS = 1000
SEED = 42
OUTPUT = "ProteinCoding_HPO_EmpiricalFDR.csv"

z = pd.read_csv(
    "Test_ForPub_NewGOHPO_FisherExact_Mis_QuantNorm_PhyloP_MafCut_AF0.1.csv"
)

terms = z["Term"].tolist()

# Match the gene universe in your original analysis
all_genes = sorted(set().union(*(set(d_HPO[t]) for t in terms)))
gene_to_idx = {g: i for i, g in enumerate(all_genes)}
n_genes = len(all_genes)

# Store each HPO term as integer gene indices
term_indices = [
    np.array(
        sorted({gene_to_idx[g] for g in d_HPO[t] if g in gene_to_idx}),
        dtype=np.int32
    )
    for t in terms
]

# -----------------------------
# Precompute variant arrays
# -----------------------------
def prepare_gene_arrays(df, genes):
    subset = df.loc[
        df["Gene"].isin(genes), ["Gene", "Statistic"]
    ].copy()

    subset["Statistic"] = pd.to_numeric(
        subset["Statistic"], errors="coerce"
    )
    subset = subset.loc[np.isfinite(subset["Statistic"])]

    grouped = {
        gene: values.to_numpy(dtype=np.float64)
        for gene, values in subset.groupby("Gene")["Statistic"]
    }

    return [
        grouped.get(g, np.empty(0, dtype=np.float64))
        for g in all_genes
    ]

fixed_arrays = prepare_gene_arrays(qm_for_fixed, all_genes)
poly_arrays = prepare_gene_arrays(qm_for_poly, all_genes)

def pooled_values(arrays, gene_indices):
    chunks = [arrays[j] for j in gene_indices if arrays[j].size]

    if not chunks:
        return np.empty(0, dtype=np.float64)

    return np.concatenate(chunks)

# -----------------------------
# Compute MWU p-values
# -----------------------------
def compute_pvalues(permutation=None):
    """
    permutation[j] is the shuffled gene-label index assigned
    to original gene j.

    Inverting that mapping gives the original genes contributing
    variants to each permuted HPO term.
    """
    inverse = (
        np.arange(n_genes)
        if permutation is None
        else np.argsort(permutation)
    )

    pvalues = np.ones(len(terms), dtype=np.float64)

    for k, members in enumerate(term_indices):
        source_genes = inverse[members]

        x = pooled_values(fixed_arrays, source_genes)
        y = pooled_values(poly_arrays, source_genes)

        if x.size == 0 or y.size == 0:
            continue

        pvalues[k] = mannwhitneyu(
            x, y,
            alternative="greater",
            method="auto"
        ).pvalue

    return pvalues

# -----------------------------
# Observed results
# -----------------------------
start = perf_counter()

observed_p = compute_pvalues()
order = np.argsort(observed_p, kind="stable")
observed_sorted = observed_p[order]

# Number of observed discoveries at each observed cutoff
n_observed = np.searchsorted(
    observed_sorted, observed_sorted, side="right"
)

# -----------------------------
# Permutations
# -----------------------------
rng = np.random.default_rng(SEED)

null_counts_sum = np.zeros(len(terms), dtype=np.float64)
null_count_matrix = np.empty(
    (N_PERMUTATIONS, len(terms)),
    dtype=np.int32
)

for b in range(N_PERMUTATIONS):
    permutation = rng.permutation(n_genes)

    perm_p = compute_pvalues(permutation)

    # Count ALL significant null HPO terms at each observed cutoff
    counts = np.searchsorted(
        np.sort(perm_p),
        observed_sorted,
        side="right"
    )

    null_count_matrix[b] = counts
    null_counts_sum += counts

    if (b + 1) % 100 == 0:
        elapsed = (perf_counter() - start) / 60
        print(
            f"{b+1}/{N_PERMUTATIONS} permutations "
            f"completed ({elapsed:.1f} min)"
        )

# -----------------------------
# Empirical FDR
# -----------------------------
mean_null = null_counts_sum / N_PERMUTATIONS

raw_fdr = np.minimum(
    1.0,
    mean_null / n_observed
)

# Monotonic adjustment:
# A more significant phenotype cannot have a larger q-value
# than a less significant phenotype.
empirical_q = np.minimum.accumulate(raw_fdr[::-1])[::-1]

# Optional: global count-based permutation p-values
global_p = (
    1 + (null_count_matrix >= n_observed).sum(axis=0)
) / (N_PERMUTATIONS + 1)

# -----------------------------
# Save results
# -----------------------------
results = pd.DataFrame({
    "Term": np.asarray(terms)[order],
    "MWU_pvalue": observed_sorted,
    "Observed_discoveries": n_observed,
    "Mean_null_discoveries": mean_null,
    "Empirical_FDR_raw": raw_fdr,
    "Empirical_FDR_monotonic": empirical_q,
    "Global_count_empirical_p": global_p
})

results.to_csv(OUTPUT, index=False)

# Save compact permutation counts for reproducibility
np.savez_compressed(
    "ProteinCoding_HPO_PermutationCounts.npz",
    observed_pvalues=observed_sorted,
    null_count_matrix=null_count_matrix,
    term_names=np.asarray(terms)[order]
)

print(f"\nCompleted in {(perf_counter()-start)/60:.2f} minutes")
print(results.head(20).to_string(index=False))


In [ ]:
v, yvalls = read_missense(path = "./", maf_cut = 0.1)
yvalls = unfold_missense(yvalls)
vv = yvalls.copy()

In [ ]:
vvk

In [ ]:
vk = v_mis[v_mis["Gene"].isin(d_HPO["Absent speech"])].copy()

vvk = vv_mis2[vv_mis2["Gene"].isin(d_HPO["Absent speech"])].copy()

#vk = vk[~vk['Gene'].isin(muc)]
#vvk = vvk[~vvk['Gene'].isin(muc)]

cuttt = 0.6

z = list(vvk["PhyloP447"])
z.sort()
cutoff = z[int(floor((len(z)*cuttt)))]


vvv = prepare_alpha(vk, vvk)
alpha = compute_alpha_cutoff(vvv, plot = True, cutoff = 2, title = "Haploinsufficient genes")
print(alpha)
print((fisher_exact(alpha[-2])[1] + fisher_exact(alpha[-3])[1])/2)

In [ ]:
pli = pd.read_csv("gnomad.v4.1.constraint_metrics.tsv", sep = "\t")
pli = pli.dropna()
pli = pli[pli["lof.pLI"] > 0.9]
keep_pli = pli["gene"]

low_mis = pli[pli["mis.z_score"] <= 2]["gene"]
high_mis = pli[pli["mis.z_score"] > 2]["gene"]

syngo = pd.read_csv("../Cell_Type_Prop/Cell_Type_Prop/syngo_genes.csv")
syngo = syngo["hgnc_symbol"]


In [ ]:
v_mis, vv_mis = read_missense(path = "./", maf_cut = 0.01)
vv_mis = unfold_missense(vv_mis)
vv_mis2 = vv_mis[(vv_mis["UnfoldedMAF"] > 0.5) & (vv_mis["UnfoldedMAF"] < 0.9)]

In [ ]:
#Stronger positive selection on haploinsufficient genes
vk = v_mis[v_mis["Gene"].isin(keep_pli)].copy()
#vk = v_mis_chp[v_mis_chp["NearestGene"].isin(keep_pli)].copy()

vvk = vv_mis2[vv_mis2["Gene"].isin(keep_pli) & (vv_mis2["Human ref"] == vv_mis2["Chimp ref"])].copy()

#vk = vk[~vk['Gene'].isin(muc)]
#vvk = vvk[~vvk['Gene'].isin(muc)]

cuttt = 0.6

z = list(vvk["PhyloP447"])
z.sort()
cutoff = z[int(floor((len(z)*cuttt)))]


vvv = prepare_alpha(vk, vvk)
alpha = compute_alpha_cutoff(vvv, plot = True, cutoff = 2, title = "Haploinsufficient genes")
print(alpha)
print((fisher_exact(alpha[-2])[1] + fisher_exact(alpha[-3])[1])/2)

In [ ]:
macse = pd.read_csv("MACSE_Results/human_derived_substitutions.tsv", sep = "\t")
macse_nsyn = macse[macse["substitution_type"] == "nonsynonymous"]


In [ ]:
vk2 = vk[vk["Position"].isin("chr" + macse_nsyn["hg38_chrom"].astype(str) + ":" + macse_nsyn["hg38_pos"].astype(str))]
vvk2 = vvk[vvk["Gene"].isin(vk2["Gene"])]

In [ ]:
vk2.shape

In [ ]:
vk2 = vk[vk["Gene"].isin(high_mis)]
vk2.shape

In [ ]:
pd.DataFrame(high_mis).to_csv("NonsynDep2_HI_Genes.txt", sep = "\t", header = False, index = False)

In [ ]:
low_mis = pli[pli["mis.z_score"] <= 2]["gene"]
high_mis = pli[pli["mis.z_score"] > 2]["gene"]

vk2 = vk[vk["Gene"].isin(high_mis)]
vvk2 = vvk[(vvk["Gene"].isin(high_mis))]
#vk2 = vk[vk["Position"].isin("chr" + macse_nsyn["hg38_chrom"].astype(str) + ":" + macse_nsyn["hg38_pos"].astype(str))]
#vk2 = vk2[vk2["Gene"].isin(high_mis)]

#vk2 = vk[vk["NearestGene"].isin(high_mis)].copy()

#vvk2 = vvk[vvk["Gene"].isin(high_mis)]
#vvk2 = vvk2[vvk2["Gene"].isin(vk["NearestGene"])]
vvv = prepare_alpha(vk2, vvk2)

cuttt = 0.6

z = list(vvk2["PhyloP447"])
z.sort()
cutoff = z[int(floor((len(z)*cuttt)))]

alpha = compute_alpha_cutoff(vvv, plot = True, cutoff = 2, title = "HI, nonsynonymous-depleted, MACSE")
print(alpha)
print((fisher_exact(alpha[-2])[1] + fisher_exact(alpha[-3])[1])/2)


In [ ]:
v = pd.read_csv("Fixed_LiangSteinNeuron.txt.gz", sep = "\t")
vv = pd.read_csv("Poly_MAF0.25_LiangSteinNeuron.txt.gz", sep = "\t")


try:
    vv = add_unfold(vv)
    vv_ref = vv[vv["Human ref"] == vv["Chimp ref"]]
    vv_alt = vv[vv["Human alt"] == vv["Chimp ref"]]
    vv_ref["fixed logfc"] = -vv_ref["logfc"].astype(float)
    vv_alt["fixed logfc"] = vv_alt["logfc"].astype(float)
    vv = pd.concat([vv_ref, vv_alt])
except:
    pass

te_blacklist = pd.read_csv("BlacklistTE_Variants.txt", sep = "\t")
    
v = v[~v["Position"].isin(te_blacklist["Position"])]
vv = vv[~vv["Position"].isin(te_blacklist["Position"])]

In [ ]:
v = pd.read_csv("HumChp_NC_Final_NoHLA_ReadFast.txt", sep = "\t")
v

In [ ]:
z = pd.read_csv("Fixed_ASCT.txt.gz", sep = "\t")
z

In [ ]:
z = z.drop(["Position.1", "PhastCons447"], axis = 1)
z

In [ ]:
z.to_csv("ASCT_Fixed_HumanDerived_Preds.csv.gz", index = False)

In [ ]:
z.columns = ["Hg38 position", "Nearest gene TSS", "Distance to nearest TSS", "PhyloP", "Number of species in alignment", "Logfc in CA", "Human allele counts", "Chimp allele counts", "Absolute logfc in CA"]
z["Logfc in CA"] = -z["Logfc in CA"]
z

In [ ]:
n = pd.read_csv("MA0113.3_NR3C1_cut-1924_hum_chp_bon_gor_org_rhe.snps.sort.txt.gz", sep  = "\t", header = None)
n

In [ ]:
n.columns = ["Hg38 position", "Human binding score", "Chimp binding score", "Bonobo binding score", "Gorilla binding score", "Orangutan binding score", "Rhesus binding score"]
n = n.set_index("Hg38 position")
n = n.replace("<200", 199)
n = n.astype(float)

In [ ]:
n["Human-chimp difference"] = n["Human binding score"] - n["Chimp binding score"]
n["Human-gorilla difference"] = n["Human binding score"] - n["Gorilla binding score"]
n

In [ ]:
z = z.set_index("Hg38 position")

In [ ]:
zn = z.join(n)
zn

In [ ]:
zn2 = zn.dropna()
zn2.to_csv("NR3C1_AllSitesJoined_ASCT.csv.gz")

In [ ]:
zn2 = zn2[(zn2["Human binding score"] >= 400) | (zn2["Chimp binding score"] >= 400)]
zn2

In [ ]:
zn2_hder = zn2[((zn2["Human-chimp difference"] < -100) & (zn2["Human-gorilla difference"] < -100)) | ((zn2["Human-chimp difference"] > 100) & (zn2["Human-gorilla difference"] > 100))]
zn2_hder.to_csv("NR3C1_ASCT_Initial_Filtered.csv")
zn2_hder[zn2_hder["Absolute logfc in CA"] > 0.1].to_csv("NR3C1_ASCT_Initial_Filtered_AbsLogfcG0.1.csv")


In [ ]:
sns.regplot(data = zn2_hder[zn2_hder["Absolute logfc in CA"] > 0.25], x = "Human-chimp difference", y = "Logfc in CA")

In [ ]:
sns.regplot(data = zn2_hder[np.abs(zn2_hder["Human-chimp difference"]) > 200], x = "Human-chimp difference", y = "Logfc in CA")

In [ ]:
x = pd.read_csv("TFBS_Results/ASCT/TFBS_SpecSup0_PS_PhastCons-1_Unpaired_MAFCut0.25_FilterNewTEs_ASCT_AbsLogfc_HrefDneCrefOnly_Top20th.txt.gz", sep = "\t")
x

In [ ]:
v = pd.read_csv("Fixed_LiangSteinNeuron.txt.gz", sep = "\t")
vv = pd.read_csv("Poly_MAF0.25_LiangSteinNeuron.txt.gz", sep = "\t")


try:
    vv = add_unfold(vv)
    vv_ref = vv[vv["Human ref"] == vv["Chimp ref"]]
    vv_alt = vv[vv["Human alt"] == vv["Chimp ref"]]
    vv_ref["fixed logfc"] = -vv_ref["logfc"].astype(float)
    vv_alt["fixed logfc"] = vv_alt["logfc"].astype(float)
    vv = pd.concat([vv_ref, vv_alt])
except:
    pass

te_blacklist = pd.read_csv("BlacklistTE_Variants.txt", sep = "\t")
    
v = v[~v["Position"].isin(te_blacklist["Position"])]
vv = vv[~vv["Position"].isin(te_blacklist["Position"])]

In [ ]:
lof = pd.read_csv("LOF_Candidates_Kern2015_hg38.bed", sep = "\t", header = None)
keep = []
for index, row in lof.iterrows():
    keep = keep + [row[0] + ":" + str(x) for x in range(row[1], row[2])]


vk = v[v["Position"].isin(keep)]
vvk = vv[vv["Position"].isin(keep)]


In [ ]:
v.sort_values("allele1_pred_counts").tail(500000)

In [ ]:
vk2 = vk[(vk["allele1_pred_counts"] > 1175) | (vk["allele2_pred_counts"] > 1175)]
vvk2 = vvk[(vvk["allele1_pred_counts"] > 1175) | (vvk["allele2_pred_counts"] > 1175)]

vk2 = vk2.copy()
vvk2 = vvk2.copy()

vvv = prepare_alpha(vk2, vvk2, stat = "abs logfc")
cutoff = 0.1
alpha = compute_alpha_cutoff(vvv, plot = True, window = [0.1, 1], cutoff = cutoff)
print(alpha)

In [ ]:
fisher_exact([[342, 1398], [122, 430]])

In [ ]:
fisher_exact([[261, 860], [28, 113]])

In [ ]:
x = pd.read_csv("Revisions_Stuff/tf_accessibility_neurodev_final.csv")

In [ ]:
v = pd.read_csv("TFBS_SpecSup0_PS_PhastCons-1_Unpaired_MAFCut0.25_FilterNewTEs_SYNGO_LiangSteinNeuron_AbsLogfc_HrefDneCrefOnly_All.txt.gz", sep = "\t")
v = v[v["Cutoff"] == 500]
v = v[v["Proportion"] == 0.6]
v = v.sort_values("Median fixed abs logfc")
vx = v[v["TF_Name"].str.upper().isin(x["gene_symbol"])].sort_values("alpha")
vx = vx[vx["Number polymorphic variants"] > 25]
vx = vx[(vx["Median fixed abs logfc"] > 0.03) | (vx["Median fixed abs logfc"] > 0.03)]
vx.to_csv("TFs_To_Pull.txt")

In [ ]:
vx["-Log$_{10}$(p-value)"] = -np.log10(vx["Fisher exact p-value"])
vx["FDR"] = fdrcorrection(vx["Fisher exact p-value"])[1]

In [ ]:
ft = pd.read_csv("fixed_TFBS_positions.tsv.gz", sep = "\t")
pt = pd.read_csv("polymorphic_TFBS_positions.tsv.gz", sep = "\t")

vt = v[v["Position"].isin(ft["Position"])]
vvt = vv[vv["Position"].isin(pt["Position"])]

vvt = vvt[vvt["Human ref"] != vvt["Chimp ref"]]


In [ ]:
syngo = pd.read_csv("../Cell_Type_Prop/Cell_Type_Prop/syngo_genes.csv")
syngo = syngo["hgnc_symbol"]

In [ ]:
vts = vt[vt["NearestGene"].isin(syngo)]
vvts = vvt[vvt["NearestGene"].isin(syngo)]
vvts = vvts[vvts["Human ref"] != vvts["Chimp ref"]]

cuttt = 0.9
z = list(vvts["abs logfc"])
z.sort()
cutoff = z[int(floor((len(z)*cuttt)))]


vvv = prepare_alpha(vts, vvts, stat = "abs logfc")
#vvv["PhyloP"] = -vvv["PhyloP"]
alpha = compute_alpha_cutoff(vvv, plot = True, window = [0, 1], cutoff = cutoff, title = "NFIB binding sites in fetal cortical exc. neurons")
plt.xlabel("Predicted absolute log$_{2}$ fold-change in CA", fontsize = 20)
plt.ylabel("Density", fontsize = 20)
print(alpha)
print((fisher_exact(alpha[-2], alternative = "greater")[1] + fisher_exact(alpha[-3], alternative = "greater")[1])/2)


In [ ]:
ft = pd.read_csv("fixed_TFBS_positions.tsv.gz", sep = "\t")
pt = pd.read_csv("polymorphic_TFBS_positions.tsv.gz", sep = "\t")

ft = ft[ft["TF_Name"] == "NFIB"]
pt = pt[pt["TF_Name"] == "NFIB"]

vt = v[v["Position"].isin(ft["Position"])]
vvt = vv[vv["Position"].isin(pt["Position"])]

vvt = vvt[vvt["Human ref"] != vvt["Chimp ref"]]


In [ ]:
c = 0
out = []
for key in d_HPO.keys():
    if c%1000 == 0:
        print(c)
    if len(d_HPO[key]) >= 20:
        vk = vt[vt["NearestGene"].isin(d_HPO[key])].copy()
        vvk = vvt[vvt["NearestGene"].isin(d_HPO[key])].copy()
        x2 = [np.float64(x) for x in list(vk["abs logfc"])]
        yvals2 = [np.float64(j) for j in list(vvk["abs logfc"])]
        yvals2.sort()
        if len(vk.index) >= 200 and len(vvk.index) >= 100:
            vvv = prepare_alpha(vk, vvk, stat = "abs logfc")
            cuttt = 0.6
            cutoff = yvals2[int(floor((len(yvals2)*cuttt)))]
            
            alpha = compute_alpha_cutoff(vvv, dn_cut = 0.01, plot = False, cutoff = cutoff)
            table1 = alpha[2]
            table2 = alpha[3]

            p = (fisher_exact(table1, alternative = "greater")[1] + fisher_exact(table2, alternative = "greater")[1])/2
            out.append([key, np.median(x2), len(x2), np.median(yvals2), len(yvals2), mwu(x2, yvals2, alternative = "greater")[1], p, alpha[0], alpha[1], table1, table2])
    c += 1
    
from statsmodels.stats.multitest import fdrcorrection


In [ ]:
df = pd.DataFrame(out)
df = df.sort_values(6)
df["FDR"] = fdrcorrection(df[6])[1]

df.columns = ["Term", "Median Fixed PhyloP", "Number Fixed Variants", "Median Polymorphic PhyloP", "Number Polymorphic Variants", "MWU p-value", "Fisher exact p-value", "alpha", "Cutoff", "[[dc1, pc1], [du1, pu1]]", "[[dc2, pc2], [du2, pu2]]", "FDR"]
df

In [ ]:
vv[(vv["SpecSup447"] >= 250) & (vv["PhyloP447"] >= 2)].sort_values("UnfoldedMAF")

In [ ]:
dff = vx.copy()

k = []
for index, row in dff.iterrows():
    if row["FDR"] < 0.05:
        k.append("FDR < 0.05")
    else:
        k.append("Not significant")
dff['Significance'] = k
fig, ax = plt.subplots(figsize=(6,4))
sns.scatterplot(data = dff, x = "alpha", y = "-Log$_{10}$(p-value)", hue = "Significance", palette = {"FDR < 0.05":"red", "Not significant":"grey"})
plt.title("Selection near synaptic genes in BS of chromatin remodeling TFs", size = 12)
plt.ylabel("-Log$_{10}$(p-value)", size = 12)
plt.xlabel("$\\alpha_{CA}$", size = 16)
#plt.xticks(size = 11)
#plt.yticks(size = 11)
plt.legend(fontsize = 12)


In [ ]:
v = pd.read_csv("TFBS_SpecSup0_PS_PhastCons-1_Unpaired_MAFCut0.25_FilterNewTEs_SYNGO_LiangSteinNeuron_AbsLogfc_HrefDneCrefOnly_All.txt.gz", sep = "\t")
v = v[v["Cutoff"] == 500]
v = v[v["Proportion"] == 0.6]

import ast
import numpy as np
from scipy.stats import fisher_exact

col = "[[dc2, du2], [pc2, pu2]]"

def get_log2_or(value):
    table = np.array(ast.literal_eval(value), dtype=float)
    odds_ratio, pvalue = fisher_exact(table)
    return np.log2(odds_ratio)

v["log2_OR"] = v[col].apply(get_log2_or)

vx = v.copy()
vx["-Log$_{10}$(p-value)"] = -np.log10(vx["Fisher exact p-value"])
vx["FDR"] = fdrcorrection(vx["Fisher exact p-value"])[1]

dff = vx.copy()

k = []
for index, row in dff.iterrows():
    if row["FDR"] < 0.05:
        k.append("FDR < 0.05")
    else:
        k.append("Not significant")
dff['Significance'] = k
fig, ax = plt.subplots(figsize=(5.6,4.2))
sns.scatterplot(data = dff, x = "log2_OR", y = "-Log$_{10}$(p-value)", hue = "Significance", palette = {"FDR < 0.05":"red", "Not significant":"grey"})
plt.title("Selection near synaptic genes TFBS", size = 14)
plt.ylabel("-Log$_{10}$(p-value)", size = 14)
plt.xlabel("Log$_{2}$(odds ratio)", size = 16)
#plt.xticks(size = 11)
#plt.yticks(size = 11)
plt.legend(fontsize = 12)


In [ ]:
lof = pd.read_csv("GOF_Candidates_Kern2015_hg38.bed", sep = "\t", header = None)
keep = []
for index, row in lof.iterrows():
    keep = keep + [row[0] + ":" + str(x) for x in range(row[1], row[2])]


vk = v[v["Position"].isin(keep)]


In [ ]:
vvk = vv[vv["Position"].isin(keep)]

In [ ]:
vk2 = vk[(vk["allele1_pred_counts"] > 500) | (vk["allele2_pred_counts"] > 500)]
vvk2 = vvk[(vvk["allele1_pred_counts"] > 500) | (vvk["allele2_pred_counts"] > 500)]

vvv = prepare_alpha(vk, vvk, stat = "abs logfc")
cutoff = 0.25
alpha = compute_alpha_cutoff(vvv, plot = True, window = [0.1, 1], cutoff = cutoff)
print(alpha)

In [ ]:
#Trying to figure out chr21
vk = v[v["SpecSup447"] > 250]
vvk = vv[vv["SpecSup447"] > 250]
vvk

In [ ]:
vk["Chrom"] = [x.split(":")[0] for x in vk["Position"]]
vvk["Chrom"] = [x.split(":")[0] for x in vvk["Position"]]

vk21 = vk[vk["Chrom"].isin(["chr21"])]
vvk21 = vvk[vvk["Chrom"].isin(["chr21"])]

In [ ]:
vkX = vk[vk["Chrom"].isin(["chr21"])]
vvk21 = vvk[vvk["Chrom"].isin(["chr21"])]

In [ ]:
fixed = vk.copy()
poly = vvk.copy()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use the same filtered noncoding tables as in the chr21 analysis.
# Both should already have the same noncoding and species-support filters.
fixed_df = fixed.copy()
poly_df = poly.copy()

# Adjust these if your existing tables use different names.
CHROM_COL = "Chrom"
PHYLOP_COL = "PhyloP447"

def summarize_chromosomes(data, variant_type):
    d = data[[CHROM_COL, PHYLOP_COL]].copy()

    d.columns = ["Chromosome", "PhyloP"]
    d["Chromosome"] = (
        d["Chromosome"].astype(str).str.replace(r"^chr", "", regex=True)
    )
    d["PhyloP"] = pd.to_numeric(d["PhyloP"], errors="coerce")

    # Keep autosomes and chrX; exclude unplaced contigs, chrY, and MT.
    chroms = [str(i) for i in range(1, 23)] + ["X"]
    d = d[d["Chromosome"].isin(chroms)]
    d = d.dropna(subset=["PhyloP"])

    result = d.groupby("Chromosome")["PhyloP"].agg(
        N="count",
        Mean="mean",
        Median="median",
        Prop_gt1=lambda x: (x > 1).mean(),
        Prop_gt2=lambda x: (x > 2).mean(),
        Prop_gt3=lambda x: (x > 3).mean()
    )

    result["Type"] = variant_type
    return result.reset_index()


fixed_stats = summarize_chromosomes(fixed_df, "Fixed")
poly_stats = summarize_chromosomes(poly_df, "Polymorphic")

chrom_stats = pd.concat(
    [fixed_stats, poly_stats],
    ignore_index=True
)

chrom_order = [str(i) for i in range(1, 23)] + ["X"]

chrom_stats["Chromosome"] = pd.Categorical(
    chrom_stats["Chromosome"],
    categories=chrom_order,
    ordered=True
)

chrom_stats = chrom_stats.sort_values(["Chromosome", "Type"])

display(chrom_stats.round(4))

In [ ]:
wide = chrom_stats.pivot(
    index="Chromosome",
    columns="Type",
    values=[
        "N", "Mean", "Median",
        "Prop_gt1", "Prop_gt2", "Prop_gt3"
    ]
)

wide.columns = [
    f"{metric}_{variant_type}"
    for metric, variant_type in wide.columns
]

wide["Mean_difference"] = (
    wide["Mean_Fixed"] - wide["Mean_Polymorphic"]
)

wide["Median_difference"] = (
    wide["Median_Fixed"] - wide["Median_Polymorphic"]
)

for threshold in [1, 2, 3]:
    wide[f"Prop_gt{threshold}_difference"] = (
        wide[f"Prop_gt{threshold}_Fixed"] -
        wide[f"Prop_gt{threshold}_Polymorphic"]
    )

wide = wide.reindex(chrom_order)

display(wide.round(4))

wide.to_csv("Chromosome_PhyloP_Fixed_vs_Polymorphic.csv")

In [ ]:
np.mean(vk21["PhyloP447"])

In [ ]:
vvk21[vvk21["PhyloP447"] > 0.5]

In [ ]:
import pandas as pd
from collections import Counter
from scipy.stats import pearsonr, spearmanr

df = pd.DataFrame({
    "poly": Counter(vvk21["NearestGene"].dropna()),
    "fixed": Counter(vk21["NearestGene"].dropna())
}).fillna(0).astype(int)

# Correlations
print("Pearson:", pearsonr(df["poly"], df["fixed"]))
print("Spearman:", spearmanr(df["poly"], df["fixed"]))

# View genes with the most substitutions
print(df.sort_values("fixed", ascending=False).head(20))

In [ ]:
df["difference"] = df["fixed"] - df["poly"]
df["ratio"] = (df["fixed"] + 0.5) / (df["poly"] + 0.5)

# Largest excess of fixed substitutions
#print(df.sort_values("difference", ascending=False).head(20))

# Largest excess of polymorphisms
print(df.sort_values("ratio", ascending = False).head(20))

In [ ]:
df.to_csv("Ratios.txt", sep = "\t")

In [ ]:
df

In [ ]:
fisher_exact([[13132, 3928], [67110 - 13132, 18816 - 3928]])

In [ ]:
np.mean(vvk21["PhyloP447"])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def sliding_window_alpha(
    vk, vvk,
    window_size=2_000_000,
    step_size=1_000_000,
    cutoff=0.5,
    min_fixed=10,
    min_poly=10,
    position_col="Position"
):
    def add_coords(df):
        df = df.copy()
        coords = df[position_col].str.rsplit(":", n=1, expand=True)
        df["chrom"] = coords[0]
        df["pos"] = coords[1].astype(int)
        return df

    fixed = add_coords(vk)
    poly = add_coords(vvk)

    results = []

    chromosomes = sorted(set(fixed["chrom"]) | set(poly["chrom"]))

    for chrom in chromosomes:
        f = fixed[fixed["chrom"] == chrom]
        p = poly[poly["chrom"] == chrom]

        max_pos = max(
            f["pos"].max() if len(f) else 0,
            p["pos"].max() if len(p) else 0
        )

        for start in range(0, int(max_pos), step_size):
            end = start + window_size

            # BED-like coordinates: start <= pos0 < end
            fw = f[(f["pos"] - 1 >= start) & (f["pos"] - 1 < end)]
            pw = p[(p["pos"] - 1 >= start) & (p["pos"] - 1 < end)]

            if len(fw) < min_fixed or len(pw) < min_poly:
                continue

            combined = prepare_alpha(
                fw.drop(columns=["chrom", "pos"]),
                pw.drop(columns=["chrom", "pos"])
            )

            # Assumes your function returns a scalar alpha
            alpha = compute_alpha_cutoff(
                combined,
                plot=False,
                cutoff=cutoff
            )
            
            results.append({
                "chrom": chrom,
                "start": start,
                "end": end,
                "midpoint": (start + end) / 2,
                "n_fixed": len(fw),
                "n_poly": len(pw),
                "alpha": alpha[0]
            })

    return pd.DataFrame(results)


alpha_windows = sliding_window_alpha(
    vk21, vvk21,
    window_size=2_000_000,
    step_size=1_000_000,
    cutoff=1
)

In [ ]:
combined = prepare_alpha(
    vk21,
    vvk21
)

# Assumes your function returns a scalar alpha
alpha = compute_alpha_cutoff(
    combined,
    plot=False,
    cutoff=2
)
alpha

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CUTOFF = 0.5
WINDOW = 2_000_000

def prepare_sites(df, kind, cutoff=CUTOFF):
    x = df[["Position", "PhyloP447"]].copy()

    coords = x["Position"].str.rsplit(":", n=1, expand=True)

    x["chrom"] = coords[0].str.replace("chr", "", regex=False)
    x["pos"] = pd.to_numeric(coords[1], errors="coerce")
    x["phyloP"] = pd.to_numeric(x["PhyloP447"], errors="coerce")

    x = x.dropna(subset=["pos", "phyloP"]).copy()
    x["pos"] = x["pos"].astype(np.int64)

    x["kind"] = kind
    x["conserved"] = x["phyloP"] > cutoff

    return x[["chrom", "pos", "kind", "conserved"]]


fixed = prepare_sites(vk, "fixed", cutoff = 1)
poly = prepare_sites(vvk, "poly", cutoff = 1)

sites = pd.concat([fixed, poly], ignore_index=True)

In [ ]:
def alpha_components(data):
    Dc = ((data["kind"] == "fixed") & data["conserved"]).sum()
    Dn = ((data["kind"] == "fixed") & ~data["conserved"]).sum()

    Pc = ((data["kind"] == "poly") & data["conserved"]).sum()
    Pn = ((data["kind"] == "poly") & ~data["conserved"]).sum()

    pc_dc = Pc / Dc if Dc > 0 else np.nan
    pn_dn = Pn / Dn if Dn > 0 else np.nan

    alpha = (
        1 - pc_dc / pn_dn
        if np.isfinite(pc_dc) and np.isfinite(pn_dn) and pn_dn > 0
        else np.nan
    )

    return {
        "Dc": Dc,
        "Dn": Dn,
        "Pc": Pc,
        "Pn": Pn,
        "Pc_Dc": pc_dc,
        "Pn_Dn": pn_dn,
        "alpha": alpha
    }

In [ ]:
def leave_one_window_out(sites, chrom="21", window=2_000_000):
    x = sites[sites["chrom"] == chrom].copy()

    full = alpha_components(x)
    full_alpha = full["alpha"]

    max_pos = int(x["pos"].max())
    results = []

    for start in range(0, max_pos, window):
        end = start + window

        # Positions are 1-based; intervals here are [start, end)
        in_window = (
            (x["pos"] - 1 >= start) &
            (x["pos"] - 1 < end)
        )

        removed = x[in_window]
        remaining = x[~in_window]

        if removed.empty:
            continue

        stats = alpha_components(remaining)
        removed_stats = alpha_components(removed)

        results.append({
            "chrom": chrom,
            "start": start,
            "end": end,
            "midpoint_mb": (start + end) / 2e6,
            "n_removed": len(removed),
            "alpha_full": full_alpha,
            "alpha_without": stats["alpha"],
            "delta_alpha": stats["alpha"] - full_alpha,
            "alpha_window": removed_stats["alpha"],
            "Pc_removed": removed_stats["Pc"],
            "Dc_removed": removed_stats["Dc"],
            "Pn_removed": removed_stats["Pn"],
            "Dn_removed": removed_stats["Dn"]
        })

    return pd.DataFrame(results), full


loo21, full21 = leave_one_window_out(sites, chrom="21")

print("Full chr21 alpha:", full21["alpha"])

print(
    loo21.sort_values(
        "delta_alpha", ascending=False
    ).head(10).to_string(index=False)
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def phyloP_bin_analysis(vk, vvk, bin_width=0.5, max_score=6):
    def prep(df, kind):
        x = df[["Position", "PhyloP447"]].copy()
        x["chrom"] = (
            x["Position"].str.split(":").str[0]
            .str.replace("chr", "", regex=False)
        )
        x["score"] = pd.to_numeric(x["PhyloP447"], errors="coerce")
        x["kind"] = kind
        return x[["chrom", "score", "kind"]].dropna()

    sites = pd.concat([
        prep(vk, "fixed"),
        prep(vvk, "poly")
    ], ignore_index=True)

    sites = sites[sites["chrom"].isin(
        [str(i) for i in range(1, 23)]
    )].copy()

    # Fine bins up to 1.5, wider bins afterward
    edges = [-np.inf, 0, 0.5, 1, 1.5, 3, 4.5, 6, np.inf]
    
    labels = [
        "<0",
        "0–0.5",
        "0.5–1",
        "1–1.5",
        "1.5–3",
        "3–4.5",
        "4.5–6",
        ">6"
    ]

    sites["bin"] = pd.cut(
        sites["score"], bins=edges,
        labels=labels, right=False
    )

    sites["group"] = np.where(sites["chrom"] == "21", "chr21", "Other autosomes")

    counts = (
        sites.groupby(["group", "bin", "kind"], observed=False)
        .size()
        .unstack("kind", fill_value=0)
        .reset_index()
    )

    counts["PD"] = counts["poly"] / counts["fixed"].replace(0, np.nan)

    # Normalize each group's P/D to its PhyloP < 0.5 reference class
    ref = (
        sites.assign(reference=sites["score"] < 0.5)
        .query("reference")
        .groupby(["group", "kind"])
        .size()
        .unstack(fill_value=0)
    )

    ref_pd = ref["poly"] / ref["fixed"]
    counts["reference_PD"] = counts["group"].map(ref_pd)
    counts["relative_PD"] = counts["PD"] / counts["reference_PD"]

    # Each bin's alpha versus the same reference
    counts["bin_alpha"] = 1 - counts["relative_PD"]

    return counts


bins = phyloP_bin_analysis(vk, vvk)

In [ ]:
def plot_bin_alpha(bins):
    x = bins.pivot(
        index="bin",
        columns="group",
        values="bin_alpha"
    )

    x.plot(
        figsize=(11, 5),
        marker="o",
        linewidth=2
    )

    plt.axhline(0, color="gray", linestyle="--", linewidth=0.8)
    plt.xlabel("PhyloP447 bin")
    plt.ylabel("Bin-specific alpha")
    plt.title("Chr21 versus other autosomes")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_bin_alpha(bins)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CUTOFF = 0.5

DIST_EDGES = [-np.inf, 10_000, 50_000, 200_000, 500_000, np.inf]
DIST_LABELS = ["0–10 kb", "10–50 kb", "50–200 kb",
               "200–500 kb", ">500 kb"]

def prepare_distance_data(vk, vvk):
    def prep(df, kind):
        x = df[["Position", "PhyloP447", "NearestDist"]].copy()

        x["chrom"] = (
            x["Position"].str.split(":").str[0]
            .str.replace("chr", "", regex=False)
        )

        x["score"] = pd.to_numeric(
            x["PhyloP447"], errors="coerce"
        )

        x["distance"] = pd.to_numeric(
            x["NearestDist"], errors="coerce"
        ).abs()

        x["kind"] = kind

        return x[["chrom", "score", "distance", "kind"]].dropna()

    sites = pd.concat([
        prep(vk, "fixed"),
        prep(vvk, "poly")
    ], ignore_index=True)

    sites = sites[
        sites["chrom"].isin([str(i) for i in range(1, 23)])
    ].copy()

    sites["conserved"] = sites["score"] > CUTOFF

    sites["distance_bin"] = pd.cut(
        sites["distance"],
        bins=DIST_EDGES,
        labels=DIST_LABELS,
        right=False
    )

    sites["group"] = np.where(
        sites["chrom"] == "21",
        "chr21",
        "Other autosomes"
    )

    return sites


sites_dist = prepare_distance_data(vk, vvk)

In [ ]:
def calculate_alpha_by_distance(sites):
    rows = []

    for (group, dist_bin), d in sites.groupby(
        ["group", "distance_bin"], observed=True
    ):
        Dc = ((d["kind"] == "fixed") & d["conserved"]).sum()
        Dn = ((d["kind"] == "fixed") & ~d["conserved"]).sum()
        Pc = ((d["kind"] == "poly") & d["conserved"]).sum()
        Pn = ((d["kind"] == "poly") & ~d["conserved"]).sum()

        alpha = (
            1 - (Pc / Dc) / (Pn / Dn)
            if min(Dc, Dn, Pc, Pn) > 0
            else np.nan
        )

        rows.append({
            "group": group,
            "distance_bin": dist_bin,
            "Dc": Dc, "Dn": Dn,
            "Pc": Pc, "Pn": Pn,
            "n_sites": len(d),
            "alpha": alpha
        })

    return pd.DataFrame(rows)


dist_results = calculate_alpha_by_distance(sites_dist)

print(dist_results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for group, d in dist_results.groupby("group"):
    d = d.set_index("distance_bin").reindex(DIST_LABELS)

    ax.plot(
        DIST_LABELS,
        d["alpha"],
        marker="o",
        linewidth=2,
        label=group
    )

ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.set_ylabel("Alpha (PhyloP447 > 0.5)")
ax.set_xlabel("Distance to nearest gene")
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_chromosome_alpha_by_distance(sites_dist):
    rows = []

    for (chrom, dist_bin), d in sites_dist.groupby(
        ["chrom", "distance_bin"], observed=True
    ):
        Dc = ((d["kind"] == "fixed") & d["conserved"]).sum()
        Dn = ((d["kind"] == "fixed") & ~d["conserved"]).sum()
        Pc = ((d["kind"] == "poly") & d["conserved"]).sum()
        Pn = ((d["kind"] == "poly") & ~d["conserved"]).sum()

        alpha = (
            1 - (Pc / Dc) / (Pn / Dn)
            if min(Dc, Dn, Pc, Pn) > 0
            else np.nan
        )

        rows.append({
            "chrom": chrom,
            "distance_bin": str(dist_bin),
            "alpha": alpha,
            "n_fixed": Dc + Dn,
            "n_poly": Pc + Pn
        })

    results = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(9, 5))

    # Plot all other autosomes in gray
    for chrom in [str(i) for i in range(1, 23) if i != 21]:
        d = (
            results[results["chrom"] == chrom]
            .set_index("distance_bin")
            .reindex(DIST_LABELS)
        )

        ax.plot(
            DIST_LABELS,
            d["alpha"],
            color="gray",
            alpha=0.35,
            linewidth=1,
            marker="o",
            markersize=3
        )

    # Highlight chromosome 21
    d21 = (
        results[results["chrom"] == "21"]
        .set_index("distance_bin")
        .reindex(DIST_LABELS)
    )

    ax.plot(
        DIST_LABELS,
        d21["alpha"],
        color="darkorange",
        linewidth=3,
        marker="o",
        markersize=7,
        label="Chr21",
        zorder=10
    )

    ax.axhline(
        0, color="black", linestyle="--",
        linewidth=0.8, alpha=0.6
    )

    ax.set_xlabel("Distance to nearest gene")
    ax.set_ylabel("Alpha (PhyloP447 > 0.5)")
    ax.set_title("Noncoding alpha by distance to nearest gene")

    ax.legend()
    ax.tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.show()

    return results


chrom_dist_results = plot_chromosome_alpha_by_distance(sites_dist)

In [ ]:
chrom_results = []

for chrom, group in sites.groupby("chrom"):
    stats = alpha_components(group)
    stats["chrom"] = chrom
    chrom_results.append(stats)

chrom_stats = pd.DataFrame(chrom_results)

chrom_stats["chrom_order"] = pd.to_numeric(
    chrom_stats["chrom"], errors="coerce"
)

chrom_stats = (
    chrom_stats.sort_values("chrom_order")
    .reset_index(drop=True)
)

print(
    chrom_stats[
        ["chrom", "Dc", "Pc", "Dn", "Pn",
         "Pc_Dc", "Pn_Dn", "alpha"]
    ].to_string(index=False)
)

In [ ]:
np.mean(alpha_windows[alpha_windows["start"] > 12000000]["alpha"])

In [ ]:
def plot_sliding_alpha(alpha_windows):
    chromosomes = sorted(alpha_windows["chrom"].unique())

    fig, axes = plt.subplots(
        len(chromosomes), 1,
        figsize=(12, max(3, 2.5 * len(chromosomes))),
        squeeze=False
    )

    for ax, chrom in zip(axes.flat, chromosomes):
        x = alpha_windows[alpha_windows["chrom"] == chrom].sort_values("midpoint")

        ax.plot(
            x["midpoint"] / 1e6,
            x["alpha"],
            marker="o",
            markersize=3,
            linewidth=1
        )

        ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
        ax.set_ylabel("Alpha")
        ax.set_title(chrom)
        ax.set_xlabel("Genomic position (Mb)")

    plt.tight_layout()
    plt.show()


plot_sliding_alpha(alpha_windows)

In [ ]:
np.mean(vk21["SpecSup447"])

In [ ]:
np.mean(vvk21["SpecSup447"])